# Jigsaw Puzzle Solver: End-to-End Reconstruction using Siamese Networks and Integer Linear Programming

This notebook demonstrates an end-to-end pipeline designed to automatically solve **4x4 jigsaw puzzles** (consisting of 16 individual tile images) using a combination of Deep Learning and Combinatorial Optimization.

### Pipeline Overview

1. **Data Loading & Preprocessing**: Loads puzzle tiles from disk, ensures color consistency (RGB conversion), and applies identical normalization (mean and standard deviation) used during model training.
2. **Siamese Neural Network (JigNet) Inference**: Performs pairwise evaluations on every potential directed combination of tiles. The model predicts the probability of one of five spatial relationships between any two tiles:
   * `No Match` (0)
   * `Left` (1)
   * `Right` (2)
   * `Up` (3)
   * `Down` (4)
3. **Pairwise Score Compilation**: Merges asymmetric edge predictions (e.g., *Tile B is Right of Tile A* and *Tile A is Left of Tile B*) to build symmetrical compatibility scores for all neighboring boundaries (`RIGHT` and `DOWN`).
4. **Integer Linear Programming (ILP)**: Models the final puzzle reconstruction as a constrained mathematical optimization problem using the `PuLP` library. The objective is to maximize the sum of spatial edge compatibility scores while strictly enforcing unique grid placements and preventing spatial overlaps.
5. **Visualization & Evaluation**: Stitches the solved tile configurations together and displays the reconstructed results side-by-side with the original reference image to evaluate pixel-wise matching accuracy.

**During inference, JigNet model expects to evaluate pairs of image tiles to predict their spatial relationships.**

## 1. Tile Tensor Specifications
**Each individual tile must be processed exactly like during training:**

*   Dimensions: Height and width of $128 \times 128$ pixels.
*   Color Channels: RGB format (3 channels).
*   Data Type: PyTorch float tensor (torch.float32).
*   Normalization: Subtracted by the training mean and divided by the training standard deviation
*   Final Shape: (3, 128, 128) for a single tile, and (B, 3, 128, 128) when batched.

## 2. Pairing Structure
Because JigNet is a Siamese Network, it accepts two inputs: tile_a and tile_b. If you have a $4 \times 4$ grid (16 tiles total), you should feed all potential combinations of adjacent tiles into the network to discover matching pairs.

**Class Label Definitions:**
*  0: No Match
*  1: tile_b is Left of tile_a
*  2: tile_b is Right of tile_a
*  3: tile_b is Up (Above) tile_a
*  4: tile_b is Down (Below) tile_a

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
JigNet + ILP jigsaw benchmark over a folder of images.

For every image in IMAGES_DIR the script:
  1. splits it into GRID_SIZE x GRID_SIZE tiles (shuffled, ground-truth positions kept in a dict)
  2. prepares the tiles for JigNet (RGB, 128x128, train mean/std normalisation)
  3. runs JigNet on all N*(N-1) directed tile pairs -> (N, N, 5) softmax probabilities
  4. builds symmetric RIGHT / DOWN edge scores
  5. solves the same ILP as in the notebook (PuLP + HiGHS/CBC)
  6. records solver status, objective value, runtime and reconstruction accuracy
Finally it prints per-image results and an aggregate summary.
"""

# ======================================================================
# CONFIGURATION  (edit these)
# ======================================================================
MODEL_CODE_DIR = "/content/drive/MyDrive/Jigsaw_Puzzle_Project/final_model"   # contains jignet_models.py
MODEL_PATH = f"{MODEL_CODE_DIR}/jignet_weights.pth"

TILE_SIZE = 128                    # JigNet input resolution
SOLVER_TIME_LIMIT = 60             # seconds per puzzle
BATCH_SIZE = 256                   # tile pairs per forward pass
SHUFFLE_TILES = True               # shuffle tile order so index carries no information
SEED = 42

# ======================================================================
# IMPORTS
# ======================================================================
import importlib.util
import os
import random
import subprocess
import sys
import time
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

if importlib.util.find_spec("pulp") is None or importlib.util.find_spec("highspy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pulp", "highspy"])

import cv2
import numpy as np
import pandas as pd
import pulp
import torch
from torchvision import transforms

# Class indices predicted by JigNet
NO_MATCH_IDX, LEFT_IDX, RIGHT_IDX, UP_IDX, DOWN_IDX = 0, 1, 2, 3, 4

# Same statistics used during JigNet training
MEAN = [0.5452349781990051, 0.6246451139450073, 0.4715678095817566]
STD = [0.23454082012176514, 0.19042210280895233, 0.21782392263412476]

preprocess = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)])

Board = List[List[Optional[int]]]
PairwiseScores = Dict[Tuple[int, int, str], float]


# ======================================================================
# MODEL
# ======================================================================
def load_jignet(model_code_dir: str, model_path: str, device: torch.device):
    """Load the trained JigNet in eval mode (mounts Drive when running on Colab)."""
    if not os.path.exists(model_code_dir):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except ImportError:
            pass
    sys.path.append(model_code_dir)
    from jignet_models import JigNet

    model = JigNet(num_classes=5).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    return model.eval()


# ======================================================================
# 1. SPLIT IMAGE INTO TILES
# ======================================================================
def split_image(img_rgb: np.ndarray, grid_size: int, tile_size: int = TILE_SIZE,
                shuffle: bool = True, rng: Optional[random.Random] = None):
    """
    Cut an RGB image into grid_size x grid_size tiles, resized to tile_size x tile_size.

    Returns:
        tiles:        list of N uint8 arrays (tile_size, tile_size, 3), in (shuffled) tile order
        gt_positions: dict {tile_index: (row, col)} = where each tile belongs in the original
    """
    h, w, _ = img_rgb.shape
    th, tw = h // grid_size, w // grid_size
    if th == 0 or tw == 0:
        raise ValueError(f"Image {w}x{h} is too small for a {grid_size}x{grid_size} grid.")

    cells = [(r, c) for r in range(grid_size) for c in range(grid_size)]
    if shuffle:
        (rng or random).shuffle(cells)

    tiles, gt_positions = [], {}
    for idx, (r, c) in enumerate(cells):
        tile = img_rgb[r * th:(r + 1) * th, c * tw:(c + 1) * tw]
        tiles.append(cv2.resize(tile, (tile_size, tile_size)))
        gt_positions[idx] = (r, c)
    return tiles, gt_positions


# ======================================================================
# 2. PREPARE TILES FOR JIGNET
# ======================================================================
def prepare_tiles(tiles: List[np.ndarray], device: torch.device) -> torch.Tensor:
    """List of RGB uint8 tiles -> normalised float32 tensor of shape (N, 3, H, W)."""
    return torch.stack([preprocess(t) for t in tiles]).to(device)


# ======================================================================
# 3. JIGNET INFERENCE ON ALL DIRECTED PAIRS
# ======================================================================
@torch.no_grad()
def predict_probs_matrix(model, tiles: torch.Tensor, batch_size: int = BATCH_SIZE) -> np.ndarray:
    """
    Returns probs_matrix of shape (N, N, 5): probs_matrix[i, j] are the softmax
    probabilities of the 5 relations of tile j with respect to tile i (zeros when i == j).
    """
    n = tiles.shape[0]
    pairs = [(i, j) for i in range(n) for j in range(n) if i != j]
    probs_matrix = np.zeros((n, n, 5), dtype=np.float32)

    for start in range(0, len(pairs), batch_size):
        chunk = pairs[start:start + batch_size]
        idx_a = torch.tensor([p[0] for p in chunk], device=tiles.device)
        idx_b = torch.tensor([p[1] for p in chunk], device=tiles.device)
        probs = torch.softmax(model(tiles[idx_a], tiles[idx_b]), dim=1).cpu().numpy()
        for (i, j), p in zip(chunk, probs):
            probs_matrix[i, j] = p
    return probs_matrix


# ======================================================================
# 4. RIGHT / DOWN EDGE SCORES
# ======================================================================
def build_pairwise_scores(probs_matrix: np.ndarray) -> PairwiseScores:
    """Average each edge with its complementary prediction (j RIGHT of i  <->  i LEFT of j)."""
    n = probs_matrix.shape[0]
    scores: PairwiseScores = {}
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            scores[(i, j, "RIGHT")] = float((probs_matrix[i, j, RIGHT_IDX] + probs_matrix[j, i, LEFT_IDX]) / 2.0)
            scores[(i, j, "DOWN")] = float((probs_matrix[i, j, DOWN_IDX] + probs_matrix[j, i, UP_IDX]) / 2.0)
    return scores


# ======================================================================
# 5. ILP SOLVER
# ======================================================================
def _new_binary(prob, name):
    """PuLP >= 4 creates variables through the problem; PuLP < 4 uses LpVariable."""
    if hasattr(prob, "add_variable"):
        return prob.add_variable(name, 0, 1, "Binary")
    return pulp.LpVariable(name, 0, 1, cat="Binary")


def _get_solver(time_limit):
    """PuLP < 4 ships CBC; PuLP >= 4 needs HiGHS (highspy)."""
    if hasattr(pulp, "PULP_CBC_CMD"):
        return pulp.PULP_CBC_CMD(msg=False, timeLimit=time_limit)
    return pulp.HiGHS(msg=False, timeLimit=time_limit)


def _solver_outcome(prob, solve_return, solve_time: float, time_limit: float) -> Tuple[str, bool]:
    """
    Returns (status string, optimality proven?) for both PuLP APIs.
      PuLP >= 4: solve() returns an LpSolveStats object with a `status` enum.
      PuLP <  4: solve() returns an int; prob.status / prob.sol_status hold the status.
    A run that stopped at the time limit never counts as proven optimal.
    """
    if hasattr(solve_return, "status") and hasattr(pulp, "LpSolveStatus"):      # PuLP >= 4
        status = getattr(solve_return.status, "name", str(solve_return.status))
        proven = solve_return.status == pulp.LpSolveStatus.Optimal
    else:                                                                       # PuLP < 4
        status = pulp.LpStatus.get(prob.status, "Unknown")
        proven = status == "Optimal"
        sol_status = getattr(prob, "sol_status", None)
        if sol_status is not None and hasattr(pulp, "LpSolutionOptimal"):
            proven = proven and sol_status == pulp.LpSolutionOptimal
    return status, proven and solve_time < time_limit


@dataclass
class SolveResult:
    board: Optional[Board]
    status: str            # solver status string (e.g. "Optimal")
    proven_optimal: bool   # True only if optimality was proven (not stopped by the time limit)
    objective: float       # total RIGHT/DOWN score of the selected arrangement
    runtime: float         # seconds: model building + solving


def board_score(board: Board, scores: PairwiseScores) -> float:
    """Sum of edge scores over all horizontal and vertical neighbours of a board."""
    rows, cols = len(board), len(board[0])
    total = 0.0
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                total += scores[(board[r][c], board[r][c + 1], "RIGHT")]
            if r + 1 < rows:
                total += scores[(board[r][c], board[r + 1][c], "DOWN")]
    return total


def solve_jigsaw_ilp(scores: PairwiseScores, grid_size: int,
                     time_limit: float = SOLVER_TIME_LIMIT) -> SolveResult:
    """Maximise the total edge score under one-tile-per-cell / one-cell-per-tile constraints."""
    R = C = grid_size
    T = grid_size * grid_size
    t0 = time.perf_counter()

    prob = pulp.LpProblem("Jigsaw_Puzzle_Solver", pulp.LpMaximize)

    def binvar(name, *idx):
        return _new_binary(prob, name + "_" + "_".join(map(str, idx)))

    X = {(i, r, c): binvar("X", i, r, c) for i in range(T) for r in range(R) for c in range(C)}
    H = {(i, j, r, c): binvar("H", i, j, r, c)
         for i in range(T) for j in range(T) if i != j for r in range(R) for c in range(C - 1)}
    V = {(i, j, r, c): binvar("V", i, j, r, c)
         for i in range(T) for j in range(T) if i != j for r in range(R - 1) for c in range(C)}

    # Objective: compatibility of all horizontal + vertical neighbours
    terms = []
    for i in range(T):
        for j in range(T):
            if i == j:
                continue
            terms += [scores[(i, j, "RIGHT")] * H[i, j, r, c] for r in range(R) for c in range(C - 1)]
            terms += [scores[(i, j, "DOWN")] * V[i, j, r, c] for r in range(R - 1) for c in range(C)]
    prob += pulp.lpSum(terms)

    # One tile per cell, one cell per tile
    for r in range(R):
        for c in range(C):
            prob += pulp.lpSum(X[i, r, c] for i in range(T)) == 1
    for i in range(T):
        prob += pulp.lpSum(X[i, r, c] for r in range(R) for c in range(C)) == 1

    # Edge variables <-> tile placement
    for r in range(R):
        for c in range(C - 1):
            for i in range(T):
                prob += pulp.lpSum(H[i, j, r, c] for j in range(T) if j != i) == X[i, r, c]
                prob += pulp.lpSum(H[j, i, r, c] for j in range(T) if j != i) == X[i, r, c + 1]
    for r in range(R - 1):
        for c in range(C):
            for i in range(T):
                prob += pulp.lpSum(V[i, j, r, c] for j in range(T) if j != i) == X[i, r, c]
                prob += pulp.lpSum(V[j, i, r, c] for j in range(T) if j != i) == X[i, r + 1, c]

    t_solve = time.perf_counter()
    solve_return = prob.solve(_get_solver(time_limit))
    solve_time = time.perf_counter() - t_solve
    status, proven = _solver_outcome(prob, solve_return, solve_time, time_limit)

    # Extract board
    board: Board = [[None] * C for _ in range(R)]
    for i in range(T):
        for r in range(R):
            for c in range(C):
                v = pulp.value(X[i, r, c])
                if v is not None and v > 0.5:
                    board[r][c] = i
    placed = [t for row in board for t in row]
    valid = None not in placed and len(set(placed)) == T

    return SolveResult(
        board=board if valid else None,
        status=status if valid else f"{status} (no valid board)",
        proven_optimal=proven and valid,
        objective=board_score(board, scores) if valid else float("nan"),
        runtime=time.perf_counter() - t0,
    )


# ======================================================================
# 6. EVALUATION
# ======================================================================
def evaluate_board(board: Optional[Board], gt_positions: Dict[int, Tuple[int, int]],
                   grid_size: int) -> Tuple[bool, int]:
    """Returns (board matches the original exactly, number of tiles in their correct cell)."""
    if board is None:
        return False, 0
    correct = sum(1 for r in range(grid_size) for c in range(grid_size)
                  if gt_positions[board[r][c]] == (r, c))
    return correct == grid_size * grid_size, correct


def gt_board(gt_positions: Dict[int, Tuple[int, int]], grid_size: int) -> Board:
    """The ground-truth arrangement expressed as a board of tile indices."""
    board: Board = [[None] * grid_size for _ in range(grid_size)]
    for tile, (r, c) in gt_positions.items():
        board[r][c] = tile
    return board


# ======================================================================
# PIPELINE
# ======================================================================
def solve_image(img_path: str, grid_size: int, model, device, rng: random.Random) -> dict:
    """Run the full pipeline on one image and return its metrics."""
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError("could not read image")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    tiles, gt_positions = split_image(img_rgb, grid_size, shuffle=SHUFFLE_TILES, rng=rng)
    tile_tensor = prepare_tiles(tiles, device)
    probs_matrix = predict_probs_matrix(model, tile_tensor)
    scores = build_pairwise_scores(probs_matrix)

    result = solve_jigsaw_ilp(scores, grid_size)
    exact, correct_tiles = evaluate_board(result.board, gt_positions, grid_size)

    return {
        "image": os.path.basename(img_path),
        "status": result.status,
        "proven_optimal": result.proven_optimal,
        "objective": result.objective,
        "gt_objective": board_score(gt_board(gt_positions, grid_size), scores),
        "runtime_s": result.runtime,
        "exact_match": exact,
        "correct_tiles": correct_tiles,
        "num_tiles": grid_size * grid_size,
    }


def run_benchmark(grid_size: int, images_dir: str) -> pd.DataFrame:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_jignet(MODEL_CODE_DIR, MODEL_PATH, device)
    rng = random.Random(SEED)

    image_paths = sorted(os.path.join(images_dir, f) for f in os.listdir(images_dir)
                         if f.lower().endswith((".png", ".jpg", ".jpeg")))
    if not image_paths:
        raise ValueError(f"No images found in {images_dir}")

    n_tiles = grid_size * grid_size
    print(f"Grid {grid_size}x{grid_size} ({n_tiles} tiles, {n_tiles * (n_tiles - 1)} JigNet pairs) "
          f"| {len(image_paths)} images | device: {device}\n")

    rows = []
    for k, path in enumerate(image_paths, 1):
        try:
            row = solve_image(path, grid_size, model, device, rng)
        except Exception as e:
            print(f"[{k}/{len(image_paths)}] {os.path.basename(path)}: ERROR ({e})")
            continue
        rows.append(row)
        print(f"[{k}/{len(image_paths)}] {row['image']}: "
              f"{'OK  ' if row['exact_match'] else 'FAIL'} | status={row['status']} "
              f"optimal={row['proven_optimal']} | obj={row['objective']:.3f} "
              f"(GT {row['gt_objective']:.3f}) | {row['runtime_s']:.2f}s | "
              f"tiles {row['correct_tiles']}/{row['num_tiles']}")

    return pd.DataFrame(rows)


def print_summary(df: pd.DataFrame, grid_size: int) -> None:
    n = len(df)
    n_tiles = grid_size * grid_size
    exact = int(df["exact_match"].sum())
    optimal = int(df["proven_optimal"].sum())
    tile_acc = 100 * df["correct_tiles"].sum() / (n * n_tiles)
    avg_time = df["runtime_s"].mean()

    print("\n" + "=" * 56)
    print(f" FINAL REPORT  |  grid {grid_size}x{grid_size} ({n_tiles} tiles)")
    print("=" * 56)
    print(f"Images processed:                 {n}")
    print(f"Perfect reconstructions:          {exact}/{n}  ({100 * exact / n:.2f}%)")
    print(f"Tile placement accuracy:          {tile_acc:.2f}%")
    print(f"Optimality proven:                {optimal}/{n}  ({100 * optimal / n:.2f}%)")
    print(f"Avg objective (selected board):   {df['objective'].mean():.3f}")
    print(f"Avg objective (ground truth):     {df['gt_objective'].mean():.3f}")
    print(f"Avg solver time per puzzle:       {avg_time:.2f}s  "
          f"(median {df['runtime_s'].median():.2f}s, max {df['runtime_s'].max():.2f}s)")
    print(f"Avg solver time per tile:         {1000 * avg_time / n_tiles:.1f} ms  "
          f"[grid {grid_size}x{grid_size}]")


### 1. Run the pipline forthe test_set and grid_size = 6

In [ ]:
# ======================================================================
# RUN
# ======================================================================
IMAGES_DIR = "/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/images/test" # adjust to your images dir
GRID_SIZE = 6                      # 3, 4, 5, 6, ...
results_df = run_benchmark(GRID_SIZE, IMAGES_DIR)
print_summary(results_df, GRID_SIZE)
results_df  # per-image table (displayed in Colab)

Grid 6x6 (36 tiles, 1260 JigNet pairs) | 65 images | device: cuda

[1/65] img_109.png: OK   | status=Optimal optimal=True | obj=55.335 (GT 55.335) | 44.04s | tiles 36/36
[2/65] img_11.png: OK   | status=Optimal optimal=True | obj=58.727 (GT 58.727) | 41.82s | tiles 36/36
[3/65] img_112.png: OK   | status=Optimal optimal=True | obj=58.561 (GT 58.561) | 41.50s | tiles 36/36
[4/65] img_132.png: OK   | status=Optimal optimal=True | obj=56.200 (GT 56.200) | 41.87s | tiles 36/36
[5/65] img_134.png: OK   | status=Optimal optimal=True | obj=58.474 (GT 58.474) | 41.06s | tiles 36/36
[6/65] img_136.png: OK   | status=Optimal optimal=True | obj=58.188 (GT 58.188) | 41.63s | tiles 36/36
[7/65] img_146.png: OK   | status=Optimal optimal=True | obj=58.106 (GT 58.106) | 41.94s | tiles 36/36
[8/65] img_159.png: OK   | status=Optimal optimal=True | obj=57.625 (GT 57.625) | 42.04s | tiles 36/36
[9/65] img_166.png: OK   | status=Optimal optimal=True | obj=59.201 (GT 59.201) | 41.88s | tiles 36/36
[10/65]

,image,status,proven_optimal,objective,gt_objective,runtime_s,exact_match,correct_tiles,num_tiles
0,img_109.png,Optimal,True,55.334929,55.334929,44.035363,True,36,36
1,img_11.png,Optimal,True,58.727088,58.727088,41.823794,True,36,36
2,img_112.png,Optimal,True,58.560830,58.560830,41.500318,True,36,36
3,img_132.png,Optimal,True,56.200306,56.200306,41.867214,True,36,36
4,img_134.png,Optimal,True,58.473578,58.473578,41.059651,True,36,36
...,...,...,...,...,...,...,...,...,...
60,img_66.png,Optimal,True,56.301526,56.301526,44.771268,True,36,36
61,img_7.png,Optimal,True,57.788318,57.788318,42.524365,True,36,36
62,img_70.png,Optimal,True,57.431709,57.431709,42.080401,True,36,36
63,img_73.png,Optimal,True,57.877745,57.877745,46.602670,True,36,36


### 1. Run the pipline forthe test_set and grid_size = 5

In [ ]:
# ======================================================================
# RUN
# ======================================================================
IMAGES_DIR = "/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/images/test"
GRID_SIZE = 5                      # 3, 4, 5, 6, ...
results_df = run_benchmark(GRID_SIZE, IMAGES_DIR)
print_summary(results_df, GRID_SIZE)
results_df  # per-image table (displayed in Colab)

Grid 5x5 (25 tiles, 600 JigNet pairs) | 65 images | device: cuda

[1/65] img_109.png: OK   | status=Optimal optimal=True | obj=36.700 (GT 36.700) | 7.31s | tiles 25/25
[2/65] img_11.png: OK   | status=Optimal optimal=True | obj=38.988 (GT 38.988) | 7.29s | tiles 25/25
[3/65] img_112.png: OK   | status=Optimal optimal=True | obj=38.387 (GT 38.387) | 7.58s | tiles 25/25
[4/65] img_132.png: OK   | status=Optimal optimal=True | obj=39.451 (GT 39.451) | 7.41s | tiles 25/25
[5/65] img_134.png: OK   | status=Optimal optimal=True | obj=39.054 (GT 39.054) | 7.48s | tiles 25/25
[6/65] img_136.png: OK   | status=Optimal optimal=True | obj=36.445 (GT 36.445) | 7.52s | tiles 25/25
[7/65] img_146.png: OK   | status=Optimal optimal=True | obj=38.293 (GT 38.293) | 7.35s | tiles 25/25
[8/65] img_159.png: OK   | status=Optimal optimal=True | obj=38.235 (GT 38.235) | 7.27s | tiles 25/25
[9/65] img_166.png: OK   | status=Optimal optimal=True | obj=38.757 (GT 38.757) | 7.30s | tiles 25/25
[10/65] img_168.p

,image,status,proven_optimal,objective,gt_objective,runtime_s,exact_match,correct_tiles,num_tiles
0,img_109.png,Optimal,True,36.700352,36.700352,7.311572,True,25,25
1,img_11.png,Optimal,True,38.987913,38.987913,7.286991,True,25,25
2,img_112.png,Optimal,True,38.387337,38.387337,7.578500,True,25,25
3,img_132.png,Optimal,True,39.451245,39.451245,7.413325,True,25,25
4,img_134.png,Optimal,True,39.054481,39.054481,7.477514,True,25,25
...,...,...,...,...,...,...,...,...,...
60,img_66.png,Optimal,True,38.251168,38.251168,7.441472,True,25,25
61,img_7.png,Optimal,True,39.337525,39.337525,7.328719,True,25,25
62,img_70.png,Optimal,True,38.414215,38.414215,7.416414,True,25,25
63,img_73.png,Optimal,True,37.859770,37.859770,7.219351,True,25,25


### 1. Run the pipline forthe test_set and grid_size = 4

In [ ]:
# ======================================================================
# RUN
# ======================================================================
IMAGES_DIR = "/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/images/test"
GRID_SIZE = 4                      # 3, 4, 5, 6, ...
results_df = run_benchmark(GRID_SIZE, IMAGES_DIR)
print_summary(results_df, GRID_SIZE)
results_df  # per-image table (displayed in Colab)

Grid 4x4 (16 tiles, 240 JigNet pairs) | 65 images | device: cuda

[1/65] img_109.png: OK   | status=Optimal optimal=True | obj=23.564 (GT 23.564) | 1.20s | tiles 16/16
[2/65] img_11.png: OK   | status=Optimal optimal=True | obj=23.447 (GT 23.447) | 1.18s | tiles 16/16
[3/65] img_112.png: OK   | status=Optimal optimal=True | obj=23.759 (GT 23.759) | 1.22s | tiles 16/16
[4/65] img_132.png: OK   | status=Optimal optimal=True | obj=23.894 (GT 23.894) | 1.23s | tiles 16/16
[5/65] img_134.png: OK   | status=Optimal optimal=True | obj=23.643 (GT 23.643) | 1.36s | tiles 16/16
[6/65] img_136.png: OK   | status=Optimal optimal=True | obj=23.858 (GT 23.858) | 1.23s | tiles 16/16
[7/65] img_146.png: OK   | status=Optimal optimal=True | obj=23.849 (GT 23.849) | 1.28s | tiles 16/16
[8/65] img_159.png: OK   | status=Optimal optimal=True | obj=23.404 (GT 23.404) | 1.21s | tiles 16/16
[9/65] img_166.png: OK   | status=Optimal optimal=True | obj=23.575 (GT 23.575) | 1.21s | tiles 16/16
[10/65] img_168.p

,image,status,proven_optimal,objective,gt_objective,runtime_s,exact_match,correct_tiles,num_tiles
0,img_109.png,Optimal,True,23.564275,23.564275,1.196255,True,16,16
1,img_11.png,Optimal,True,23.446818,23.446818,1.179375,True,16,16
2,img_112.png,Optimal,True,23.758671,23.758671,1.216517,True,16,16
3,img_132.png,Optimal,True,23.894329,23.894329,1.234299,True,16,16
4,img_134.png,Optimal,True,23.642889,23.642889,1.360323,True,16,16
...,...,...,...,...,...,...,...,...,...
60,img_66.png,Optimal,True,22.410372,22.410372,1.427806,True,16,16
61,img_7.png,Optimal,True,23.234580,23.234580,1.202183,True,16,16
62,img_70.png,Optimal,True,23.729113,23.729113,1.208887,True,16,16
63,img_73.png,Optimal,True,23.248655,23.248655,1.223141,True,16,16


### 1. Run the pipline forthe test_set and grid_size = 3

In [ ]:
# ======================================================================
# RUN
# ======================================================================
IMAGES_DIR = "/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/images/test"
GRID_SIZE = 3                      # 3, 4, 5, 6, ...
results_df = run_benchmark(GRID_SIZE, IMAGES_DIR)
print_summary(results_df, GRID_SIZE)
results_df  # per-image table (displayed in Colab)

Grid 3x3 (9 tiles, 72 JigNet pairs) | 65 images | device: cuda

[1/65] img_109.png: OK   | status=Optimal optimal=True | obj=11.624 (GT 11.624) | 0.27s | tiles 9/9
[2/65] img_11.png: OK   | status=Optimal optimal=True | obj=11.823 (GT 11.823) | 0.28s | tiles 9/9
[3/65] img_112.png: OK   | status=Optimal optimal=True | obj=11.952 (GT 11.952) | 0.25s | tiles 9/9
[4/65] img_132.png: OK   | status=Optimal optimal=True | obj=10.859 (GT 10.859) | 0.27s | tiles 9/9
[5/65] img_134.png: OK   | status=Optimal optimal=True | obj=11.677 (GT 11.677) | 0.24s | tiles 9/9
[6/65] img_136.png: OK   | status=Optimal optimal=True | obj=11.743 (GT 11.743) | 0.24s | tiles 9/9
[7/65] img_146.png: OK   | status=Optimal optimal=True | obj=11.925 (GT 11.925) | 0.24s | tiles 9/9
[8/65] img_159.png: OK   | status=Optimal optimal=True | obj=11.278 (GT 11.278) | 0.26s | tiles 9/9
[9/65] img_166.png: OK   | status=Optimal optimal=True | obj=11.700 (GT 11.700) | 0.27s | tiles 9/9
[10/65] img_168.png: OK   | status=Op

,image,status,proven_optimal,objective,gt_objective,runtime_s,exact_match,correct_tiles,num_tiles
0,img_109.png,Optimal,True,11.624115,11.624115,0.266717,True,9,9
1,img_11.png,Optimal,True,11.822759,11.822759,0.282797,True,9,9
2,img_112.png,Optimal,True,11.951503,11.951503,0.247985,True,9,9
3,img_132.png,Optimal,True,10.859438,10.859438,0.266646,True,9,9
4,img_134.png,Optimal,True,11.677006,11.677006,0.244267,True,9,9
...,...,...,...,...,...,...,...,...,...
60,img_66.png,Optimal,True,11.614082,11.614082,0.241639,True,9,9
61,img_7.png,Optimal,True,11.806161,11.806161,0.239869,True,9,9
62,img_70.png,Optimal,True,11.803060,11.803060,0.240115,True,9,9
63,img_73.png,Optimal,True,11.879221,11.879221,0.243491,True,9,9


### 1. Run the pipline forthe COCO_set and grid_size = 4

In [ ]:
# ======================================================================
# RUN
# ======================================================================
IMAGES_DIR = "/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/COCO/resized_images"
GRID_SIZE = 4                      # 3, 4, 5, 6, ...
results_df = run_benchmark(GRID_SIZE, IMAGES_DIR)
print_summary(results_df, GRID_SIZE)
results_df  # per-image table (displayed in Colab)


Grid 4x4 (16 tiles, 240 JigNet pairs) | 250 images | device: cuda

[1/250] 000000007386.jpg: OK   | status=Optimal optimal=True | obj=22.255 (GT 22.255) | 1.24s | tiles 16/16
[2/250] 000000011051.jpg: OK   | status=Optimal optimal=True | obj=21.282 (GT 21.282) | 1.22s | tiles 16/16
[3/250] 000000011197.jpg: OK   | status=Optimal optimal=True | obj=21.683 (GT 21.683) | 2.40s | tiles 16/16
[4/250] 000000011760.jpg: FAIL | status=Optimal optimal=True | obj=19.008 (GT 18.701) | 10.08s | tiles 6/16
[5/250] 000000015751.jpg: OK   | status=Optimal optimal=True | obj=22.447 (GT 22.447) | 1.22s | tiles 16/16
[6/250] 000000023899.jpg: FAIL | status=Optimal optimal=True | obj=15.158 (GT 13.856) | 8.08s | tiles 0/16
[7/250] 000000025986.jpg: OK   | status=Optimal optimal=True | obj=23.766 (GT 23.766) | 1.21s | tiles 16/16
[8/250] 000000030504.jpg: OK   | status=Optimal optimal=True | obj=23.040 (GT 23.040) | 1.26s | tiles 16/16
[9/250] 000000031269.jpg: OK   | status=Optimal optimal=True | obj=21.

,image,status,proven_optimal,objective,gt_objective,runtime_s,exact_match,correct_tiles,num_tiles
0,000000007386.jpg,Optimal,True,22.254765,22.254765,1.239265,True,16,16
1,000000011051.jpg,Optimal,True,21.281996,21.281996,1.223571,True,16,16
2,000000011197.jpg,Optimal,True,21.682515,21.682515,2.399586,True,16,16
3,000000011760.jpg,Optimal,True,19.008066,18.701476,10.078813,False,6,16
4,000000015751.jpg,Optimal,True,22.447081,22.447081,1.224704,True,16,16
...,...,...,...,...,...,...,...,...,...
245,000000573094.jpg,Optimal,True,21.514895,21.514895,1.239299,True,16,16
246,000000577539.jpg,Optimal,True,23.600851,23.600851,1.242871,True,16,16
247,000000578500.jpg,Optimal,True,23.355982,23.355982,1.239797,True,16,16
248,000000579635.jpg,Optimal,True,22.224173,22.224173,1.250171,True,16,16


### 1. Run the pipline forthe COCO_set and grid_size = 6

In [ ]:
# ======================================================================
# RUN
# ======================================================================
IMAGES_DIR = "/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/COCO/resized_images"
GRID_SIZE = 6                      # 3, 4, 5, 6, ...
results_df = run_benchmark(GRID_SIZE, IMAGES_DIR)
print_summary(results_df, GRID_SIZE)
results_df  # per-image table (displayed in Colab)

Grid 6x6 (36 tiles, 1260 JigNet pairs) | 250 images | device: cpu

[1/250] 000000007386.jpg: FAIL | status=TimeLimit optimal=False | obj=9.296 (GT 53.575) | 102.67s | tiles 2/36
[2/250] 000000011051.jpg: FAIL | status=TimeLimit optimal=False | obj=6.146 (GT 53.534) | 103.89s | tiles 5/36
[3/250] 000000011197.jpg: FAIL | status=TimeLimit optimal=False | obj=5.356 (GT 53.367) | 103.28s | tiles 0/36
[4/250] 000000011760.jpg: FAIL | status=TimeLimit optimal=False | obj=15.493 (GT 49.738) | 106.70s | tiles 7/36
[5/250] 000000015751.jpg: OK   | status=Optimal optimal=False | obj=58.484 (GT 58.484) | 94.29s | tiles 36/36
[6/250] 000000023899.jpg: FAIL | status=TimeLimit optimal=False | obj=15.307 (GT 34.475) | 104.20s | tiles 0/36
[7/250] 000000025986.jpg: OK   | status=Optimal optimal=False | obj=57.745 (GT 57.745) | 62.41s | tiles 36/36
[8/250] 000000030504.jpg: OK   | status=Optimal optimal=False | obj=55.967 (GT 55.967) | 91.87s | tiles 36/36
[9/250] 000000031269.jpg: FAIL | status=TimeLi

,image,status,proven_optimal,objective,gt_objective,runtime_s,exact_match,correct_tiles,num_tiles
0,000000007386.jpg,TimeLimit,False,9.296329,53.574586,102.665876,False,2,36
1,000000011051.jpg,TimeLimit,False,6.146321,53.533681,103.891937,False,5,36
2,000000011197.jpg,TimeLimit,False,5.355806,53.366748,103.281359,False,0,36
3,000000011760.jpg,TimeLimit,False,15.493276,49.738409,106.702338,False,7,36
4,000000015751.jpg,Optimal,False,58.484000,58.484000,94.289400,True,36,36
...,...,...,...,...,...,...,...,...,...
245,000000573094.jpg,Optimal,False,56.858597,56.858597,63.563723,True,36,36
246,000000577539.jpg,Optimal,False,55.902256,55.902256,75.261773,True,36,36
247,000000578500.jpg,Optimal,False,57.368822,57.368822,64.993643,True,36,36
248,000000579635.jpg,TimeLimit,False,14.345074,48.388516,103.678082,False,2,36
